# Валидация моделей прогноза качества (AGENT-06)

## Методология валидации:
1. **Разделение по времени (Strict Out-of-Time)**:
   - Обучающая выборка (Train): `2023-01-01` – `2025-12-31`
   - Тестовая выборка (Test): `2026-01-01` – `2026-08-07`
   - Исключена утечка данных будущего (random shuffle = False).

2. **Walk-forward кросс-валидация**:
   - 5 временных срезов (фолдов) с шагом +6 месяцев.

3. **Метрики качества**:
   - MAE (Mean Absolute Error)
   - RMSE (Root Mean Squared Error)
   - Bias (Mean Error / среднее систематическое смещение)
   - Процент улучшения точности относительно базового виртуального анализатора (ВАК).

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# Добавляем корень проекта
sys.path.insert(0, str(Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()))

from src.agents.quality_agent import QualityAgent
from src.utils.vac_formulas import (
    vac_sulfur_24_2000,
    vac_d15_godt,
    vac_t50_godt,
    vac_t95_godt,
    vac_cfpp_godt
)

In [ ]:
# Загрузка телеметрии и реальных замеров качества
telem = pd.read_parquet('data/processed/telemetry_clean.parquet')
telem['date'] = pd.to_datetime(telem['date']).dt.tz_localize(None)
telem = telem.sort_values('date').reset_index(drop=True)

qual = pd.read_parquet('data/processed/telemetry_with_quality.parquet')
qual['date'] = pd.to_datetime(qual['date']).dt.tz_localize(None)

tag_map = {
    'Mg.Sulfur': 'Sulfur',
    'D15': 'D15',
    '50%.T': 'T50',
    '90%.T': 'T90',
    '95%.T': 'T95',
    'CFPP': 'CFPP'
}
qual_filtered = qual[qual['tag'].isin(tag_map.keys())]
qual_pivot = qual_filtered.pivot(index='date', columns='tag', values='value_quality').rename(columns=tag_map)

merged = pd.merge(telem, qual_pivot, on='date', how='left')
print(f"Объединённый датасет: {merged.shape}")

In [ ]:
# Расчёт базовых прогнозов ВАК и гибридной модели
agent = QualityAgent(model_path='output/models/quality_lgbm.pkl')

merged['pred_vac_Sulfur'] = vac_sulfur_24_2000(merged)
merged['pred_ml_Sulfur'] = agent.predict(merged, vac=merged['pred_vac_Sulfur'])
merged['pred_vac_D15'] = vac_d15_godt(merged)
merged['pred_vac_T50'] = vac_t50_godt(merged)
merged['pred_vac_T95'] = vac_t95_godt(merged)
merged['pred_vac_T90'] = np.maximum(merged['pred_vac_T50'], merged['pred_vac_T95'] - 12.0)
merged['pred_vac_CFPP'] = vac_cfpp_godt(merged)

# Разделение Train/Test по времени
train_df = merged[(merged['date'] >= '2023-01-01') & (merged['date'] < '2026-01-01')]
test_df = merged[(merged['date'] >= '2026-01-01') & (merged['date'] <= '2026-08-07')]
print(f"Train период: {len(train_df)} точек, Test период: {len(test_df)} точек")

In [ ]:
# Метрики на тесте 2026 года
def calc_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    yt, yp = y_true[mask], y_pred[mask]
    return {
        'MAE': mean_absolute_error(yt, yp),
        'RMSE': np.sqrt(np.mean((yt - yp)**2)),
        'bias': np.mean(yp - yt)
    }

s_ml = calc_metrics(test_df['Sulfur'].values, test_df['pred_ml_Sulfur'].values)
s_vac = calc_metrics(test_df['Sulfur'].values, test_df['pred_vac_Sulfur'].values)
improvement = (1.0 - s_ml['MAE'] / s_vac['MAE']) * 100

print(f"Sulfur Test 2026:")
print(f"  Гибридная ML-модель: MAE={s_ml['MAE']:.3f}, RMSE={s_ml['RMSE']:.3f}, bias={s_ml['bias']:.3f}")
print(f"  Чистый ВАК baseline: MAE={s_vac['MAE']:.3f}, RMSE={s_vac['RMSE']:.3f}, bias={s_vac['bias']:.3f}")
print(f"  Снижение ошибки: {improvement:.1f}%")

In [ ]:
# Загрузка и вывод итогового validation_report.json
report_path = Path('output/validation_report.json')
if report_path.exists():
    with open(report_path) as f:
        report_data = json.load(f)
    print("\nСводка Walk-forward фолдов:")
    for fold in report_data.get('walk_forward_folds', []):
        print(f"Fold {fold['fold']}: {fold['test_period']} -> Sulfur ML MAE={fold['metrics']['Sulfur_ML_MAE']:.3f}")